# pLM Covariance Pooling — Embedding Extraction

**PP1 SoSe2026 | TU München**  
Team: Joel Simon, Julius Schmidt, Lisa Börner, Andreas Weitz

Dieses Notebook führt den GPU-intensiven Schritt aus: per-Residue Embeddings aus **ProtT5** extrahieren und als HDF5 cachen.  
Das Modell wird direkt von HuggingFace geladen — kein lokaler Checkpoint nötig.

### Reihenfolge
1. GPU Check
2. Google Drive mounten
3. **Daten herunterladen** (DeepLoc + Meltome)
4. Configuration
5. Repo installieren
6. Modell testen
7. Testlauf 100 Sequenzen
8. Voller Run

## 1 · GPU Check

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU found. Go to Runtime -> Change runtime type -> GPU and restart.'
    )

gpu = torch.cuda.get_device_properties(0)
print(f'GPU:  {gpu.name}')
print(f'VRAM: {gpu.total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

## 2 · Google Drive mounten

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print('Drive mounted. Top-level folders:')
print(os.listdir('/content/drive/MyDrive'))

## 3 · Daten herunterladen

Lädt **DeepLoc** (Subcellular Localisation, 10 Klassen) und **Meltome** (Thermostabilität, Regression) herunter und konvertiert sie in das benötigte Format:

- `train.fasta` / `test.fasta` — Proteinsequenzen
- `train_labels.csv` / `test_labels.csv` — zwei Spalten: `id`, `label`

**DeepLoc** kommt von [bio_embeddings](https://github.com/sacdallago/bio_embeddings) (Standard-Benchmark-Split aus dem ProtT5-Paper).  
**Meltome** kommt von [FLIP](https://github.com/J-SNACKKB/FLIP) (Fitness Landscape Inference for Proteins).

In [ ]:
from pathlib import Path
import subprocess, sys

# Zielordner auf Drive
DRIVE_ROOT = Path('/content/drive/MyDrive/pp1_sop')
RAW_ROOT   = DRIVE_ROOT / 'raw'

(RAW_ROOT / 'deeploc').mkdir(parents=True, exist_ok=True)
(RAW_ROOT / 'meltome').mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests'], check=True)
import requests
print('Setup abgeschlossen.')

In [ ]:
# ============================================================
# DeepLoc
# Quelle: bio_embeddings Benchmark-Daten (oeffentlich)
# Header-Format: >PROTEIN_ID SUBLOC_CLASS PARTITION
# Beispiel: >P12345 Nucleus train
# ============================================================
import requests, textwrap
from pathlib import Path

DEEPLOC_FASTA_URL = (
    'http://data.bioembeddings.com/public/embeddings/'
    'benchmarks/deeploc/sequences.fasta'
)

raw_fasta = Path('/content/deeploc_raw.fasta')

print('Lade DeepLoc FASTA...')
r = requests.get(DEEPLOC_FASTA_URL, timeout=120)
if r.status_code != 200:
    raise RuntimeError(
        f'Download fehlgeschlagen (HTTP {r.status_code}).\n'
        'Bitte DeepLoc manuell von https://services.healthtech.dtu.dk/services/DeepLoc-1.0/ '
        'herunterladen und als /content/deeploc_raw.fasta hochladen.'
    )
raw_fasta.write_bytes(r.content)
print(f'Gespeichert: {raw_fasta} ({raw_fasta.stat().st_size / 1e6:.1f} MB)')

# --- Parsen und in train/test aufteilen ---
def parse_deeploc_fasta(path: Path):
    # Erwartet Header: >ID LABEL PARTITION
    # Partition ist 'train' oder 'test' (manchmal auch 'val')
    train_seqs, test_seqs = [], []
    train_labels, test_labels = [], []
    seq_id = label = partition = None
    parts = []
    with open(path) as fh:
        for line in fh:
            line = line.rstrip()
            if not line:
                continue
            if line.startswith('>'):
                if seq_id is not None:
                    seq = ''.join(parts)
                    if partition in ('test', 'val'):
                        test_seqs.append((seq_id, seq))
                        test_labels.append((seq_id, label))
                    else:
                        train_seqs.append((seq_id, seq))
                        train_labels.append((seq_id, label))
                fields = line[1:].split()
                seq_id    = fields[0] if len(fields) > 0 else f'seq{len(train_seqs)}'
                label     = fields[1] if len(fields) > 1 else 'Unknown'
                partition = fields[2].lower() if len(fields) > 2 else 'train'
                parts = []
            else:
                parts.append(line)
    if seq_id is not None:
        seq = ''.join(parts)
        if partition in ('test', 'val'):
            test_seqs.append((seq_id, seq))
            test_labels.append((seq_id, label))
        else:
            train_seqs.append((seq_id, seq))
            train_labels.append((seq_id, label))
    return train_seqs, train_labels, test_seqs, test_labels

def write_fasta(records, path: Path):
    with open(path, 'w') as fh:
        for seq_id, seq in records:
            fh.write(f'>{seq_id}\n')
            for chunk in textwrap.wrap(seq, 60):
                fh.write(chunk + '\n')

def write_label_csv(labels, path: Path):
    with open(path, 'w') as fh:
        fh.write('id,label\n')
        for seq_id, label in labels:
            fh.write(f'{seq_id},{label}\n')

tr_seqs, tr_labels, te_seqs, te_labels = parse_deeploc_fasta(raw_fasta)

dl_dir = RAW_ROOT / 'deeploc'
write_fasta(tr_seqs, dl_dir / 'train.fasta')
write_fasta(te_seqs, dl_dir / 'test.fasta')
write_label_csv(tr_labels, dl_dir / 'train_labels.csv')
write_label_csv(te_labels, dl_dir / 'test_labels.csv')

# Eindeutige Labels anzeigen
unique_labels = sorted({l for _, l in tr_labels})
print(f'Train: {len(tr_seqs)} Sequenzen | Test: {len(te_seqs)} Sequenzen')
print(f'Klassen ({len(unique_labels)}): {unique_labels}')
print('DeepLoc-Daten gespeichert.')

In [ ]:
# ============================================================
# Meltome (FLIP Benchmark)
# Quelle: FLIP GitHub (github.com/J-SNACKKB/FLIP)
# Format: CSV mit Spalten 'sequence', 'target' (Schmelztemperatur), 'set'
# ============================================================
import requests, textwrap, hashlib
import csv as csv_mod
from pathlib import Path
from io import StringIO

MELTOME_TRAIN_URL = (
    'https://raw.githubusercontent.com/J-SNACKKB/FLIP/'
    'main/splits/meltome/splits/mixed_split/train.csv'
)
MELTOME_TEST_URL = (
    'https://raw.githubusercontent.com/J-SNACKKB/FLIP/'
    'main/splits/meltome/splits/mixed_split/test.csv'
)

def download_text(url):
    r = requests.get(url, timeout=120)
    if r.status_code != 200:
        raise RuntimeError(
            f'Download fehlgeschlagen: {url} (HTTP {r.status_code})\n'
            'Bitte Meltome CSV manuell von github.com/J-SNACKKB/FLIP herunterladen.'
        )
    return r.text

def parse_flip_csv(text):
    # FLIP Meltome hat Spalten: sequence, target (und moeglicherweise weitere)
    # Wir generieren eine ID aus dem MD5-Hash der Sequenz
    reader = csv_mod.DictReader(StringIO(text))
    rows = list(reader)
    # Normalisiere Spaltennamen (FLIP verwendet manchmal 'sequence' oder 'aa_seq')
    seq_col = next((c for c in rows[0] if 'seq' in c.lower()), None)
    tgt_col = next((c for c in rows[0] if 'target' in c.lower() or 'tm' in c.lower()), None)
    if seq_col is None or tgt_col is None:
        raise ValueError(f'Unbekannte Spalten: {list(rows[0].keys())}')
    seqs, labels = [], []
    for i, row in enumerate(rows):
        seq = row[seq_col].strip().upper()
        tgt = row[tgt_col].strip()
        if not seq or not tgt:
            continue
        seq_id = f'melt_{i:05d}'
        seqs.append((seq_id, seq))
        labels.append((seq_id, tgt))
    return seqs, labels

print('Lade Meltome Train...')
train_text = download_text(MELTOME_TRAIN_URL)
print('Lade Meltome Test...')
test_text  = download_text(MELTOME_TEST_URL)

tr_seqs, tr_labels = parse_flip_csv(train_text)
te_seqs, te_labels = parse_flip_csv(test_text)

melt_dir = RAW_ROOT / 'meltome'
write_fasta(tr_seqs, melt_dir / 'train.fasta')
write_fasta(te_seqs, melt_dir / 'test.fasta')
write_label_csv(tr_labels, melt_dir / 'train_labels.csv')
write_label_csv(te_labels, melt_dir / 'test_labels.csv')

sample_temps = [float(l) for _, l in tr_labels[:5]]
print(f'Train: {len(tr_seqs)} Proteine | Test: {len(te_seqs)} Proteine')
print(f'Beispiel-Temperaturen (Train): {sample_temps}')
print('Meltome-Daten gespeichert.')

In [ ]:
# Kurzer Sanity-Check: alle 8 Dateien vorhanden?
expected = [
    'deeploc/train.fasta', 'deeploc/test.fasta',
    'deeploc/train_labels.csv', 'deeploc/test_labels.csv',
    'meltome/train.fasta', 'meltome/test.fasta',
    'meltome/train_labels.csv', 'meltome/test_labels.csv',
]
print(f'{"Datei":<40} Status')
print('-' * 50)
all_ok = True
for f in expected:
    p = RAW_ROOT / f
    ok = p.exists() and p.stat().st_size > 0
    print(f'{f:<40} {"OK" if ok else "FEHLT"}')
    all_ok = all_ok and ok
if all_ok:
    print('\nAlle Dateien vorhanden. Weiter mit Zelle 4.')
else:
    print('\nEinige Dateien fehlen — Download oben wiederholen oder manuell hochladen.')

## 4 · Configuration

Pfade und Modell — normalerweise nichts anpassen nötig wenn Zelle 3 durchgelaufen ist.

In [ ]:
from pathlib import Path

# --- Hier anpassen falls noetig ----------------------------------------------
DRIVE_ROOT = Path('/content/drive/MyDrive/pp1_sop')
RAW_ROOT   = DRIVE_ROOT / 'raw'
EMBED_OUT  = DRIVE_ROOT / 'embeddings'

# ProtT5 direkt von HuggingFace -- kein lokaler Checkpoint noetig
MODEL_ID   = 'Rostlab/prot_t5_xl_uniref50-enc'

BATCH_SIZE = 4      # auf 2 oder 1 reduzieren bei CUDA OOM
DEVICE     = 'cuda'
# -----------------------------------------------------------------------------

EMBED_OUT.mkdir(parents=True, exist_ok=True)

print(f'Modell: {MODEL_ID}')
print(f'Output: {EMBED_OUT}')

## 5 · Repo installieren

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_ORG/pLM-covariance-pooling.git'  # <- anpassen
REPO_DIR = Path('/content/pLM-covariance-pooling')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[extract]'],
    cwd=str(REPO_DIR), check=True,
)
print('Installation abgeschlossen.')

## 6 · Modell laden und testen

ProtT5 wird beim ersten Aufruf von HuggingFace heruntergeladen (~3 GB).

In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR / 'src'))

import torch
from transformers import T5EncoderModel, T5Tokenizer

print(f'Lade Tokenizer ({MODEL_ID})...')
tokenizer = T5Tokenizer.from_pretrained(MODEL_ID, do_lower_case=False)

print('Lade Model (erster Start: ~3 GB Download)...')
model = T5EncoderModel.from_pretrained(MODEL_ID)
model.eval().to(DEVICE)
for p in model.parameters():
    p.requires_grad_(False)

n_layers = model.config.num_layers
d_model  = model.config.d_model
print(f'Layers: {n_layers}  |  Hidden dim d: {d_model}')

test_seq = 'MKTLLLTLVVVTIVCLDLGAVGNK'
enc = tokenizer(
    [' '.join(list(test_seq))],
    add_special_tokens=True, return_tensors='pt'
).to(DEVICE)
with torch.inference_mode():
    out = model(**enc)
emb = out.last_hidden_state[0, :len(test_seq), :]
print(f'Smoke-Test OK: L={len(test_seq)} -> Embedding {tuple(emb.shape)}')
assert emb.shape == (len(test_seq), d_model)

del model, tokenizer, enc, out, emb
torch.cuda.empty_cache()
print('Modell entladen, GPU-Speicher freigegeben.')

## 7 · Hilfsfunktion

In [ ]:
import subprocess, sys

def run_extraction(fasta, output, layers=None, batch_size=BATCH_SIZE):
    cmd = [
        sys.executable,
        str(REPO_DIR / 'scripts' / 'extract_embeddings.py'),
        '--sequences', str(fasta),
        '--model',     MODEL_ID,
        '--output',    str(output),
        '--batch-size', str(batch_size),
        '--device',    DEVICE,
    ]
    if layers:
        cmd += ['--layers'] + layers
    print(f'Starte: {fasta.name} -> {output.name}')
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, cwd=str(REPO_DIR)
    )
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Extraktion fehlgeschlagen (exit {proc.returncode})')
    print(f'Fertig -> {output}\n')

## 8 · Testlauf mit 100 Sequenzen

**Immer zuerst ausführen** bevor der volle Datensatz läuft.

In [ ]:
import textwrap
from pathlib import Path

def subsample_fasta(src: Path, dst: Path, n: int = 100):
    records, seq_id, parts = [], None, []
    with open(src) as fh:
        for line in fh:
            line = line.rstrip()
            if not line:
                continue
            if line.startswith('>'):
                if seq_id is not None:
                    records.append((seq_id, ''.join(parts)))
                seq_id, parts = line, []
            else:
                parts.append(line)
    if seq_id is not None:
        records.append((seq_id, ''.join(parts)))
    records = records[:n]
    dst.parent.mkdir(parents=True, exist_ok=True)
    with open(dst, 'w') as fh:
        for header, seq in records:
            fh.write(header + '\n')
            for chunk in textwrap.wrap(seq, 60):
                fh.write(chunk + '\n')
    print(f'Subsampled {len(records)} Sequenzen -> {dst}')

SCRATCH     = Path('/content/test_run')
test_fasta  = SCRATCH / 'deeploc_100.fasta'
test_output = SCRATCH / 'deeploc_100.h5'

subsample_fasta(RAW_ROOT / 'deeploc' / 'train.fasta', test_fasta, n=100)

In [ ]:
run_extraction(fasta=test_fasta, output=test_output, batch_size=4)

In [ ]:
import h5py

with h5py.File(test_output, 'r') as f:
    ids  = list(f.keys())
    emb0 = f[ids[0]]['embeddings'][:]

print(f'Proteine im Test-HDF5: {len(ids)}')
print(f'Erstes Embedding: id={ids[0]!r}  shape={emb0.shape}  dtype={emb0.dtype}')
assert emb0.ndim == 2 and emb0.shape[1] == 1024, 'Unerwartete Embedding-Dimension!'
print('Testlauf erfolgreich. Voller Run kann starten.')

## 9 · Voller Run — alle vier Dataset-Splits

Output geht direkt nach Google Drive.

**Typische Laufzeiten auf T4:**
- DeepLoc train (~8k Proteine): ~15–20 min
- DeepLoc test (~2k Proteine): ~4–5 min

> Bei `CUDA out of memory`: `BATCH_SIZE` in Zelle 4 auf 2 oder 1 reduzieren.

In [ ]:
run_extraction(
    fasta  = RAW_ROOT / 'deeploc' / 'train.fasta',
    output = EMBED_OUT / 'deeploc_train.h5',
)

In [ ]:
run_extraction(
    fasta  = RAW_ROOT / 'deeploc' / 'test.fasta',
    output = EMBED_OUT / 'deeploc_test.h5',
)

In [ ]:
run_extraction(
    fasta  = RAW_ROOT / 'meltome' / 'train.fasta',
    output = EMBED_OUT / 'meltome_train.h5',
)

In [ ]:
run_extraction(
    fasta  = RAW_ROOT / 'meltome' / 'test.fasta',
    output = EMBED_OUT / 'meltome_test.h5',
)

## 10 · Output prüfen

In [ ]:
import h5py

expected = ['deeploc_train.h5', 'deeploc_test.h5', 'meltome_train.h5', 'meltome_test.h5']

print(f'{"Datei":<30} {"Vorhanden":<12} {"Proteine":<12} {"Groesse (MB)"}')
print('-' * 65)
for fname in expected:
    p = EMBED_OUT / fname
    if p.exists():
        size_mb = p.stat().st_size / 1e6
        with h5py.File(p, 'r') as f:
            n = len(f.keys())
        print(f'{fname:<30} {"JA":<12} {n:<12} {size_mb:.1f}')
    else:
        print(f'{fname:<30} {"FEHLT":<12}')

## 11 · (Optional) Layer Sweep

In [ ]:
# ProtT5-XL hat 24 Transformer-Schichten
SWEEP_LAYERS = ['last', '6', '12', '18']

run_extraction(
    fasta  = RAW_ROOT / 'deeploc' / 'train.fasta',
    output = EMBED_OUT / 'deeploc_train.h5',
    layers = SWEEP_LAYERS,
)

## 12 · (Optional) Unsupervised Covariance Pooler trainieren

In [ ]:
import subprocess, sys

MODELS_DIR = DRIVE_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)
DC_VALUES  = [8, 16, 24, 32, 48]

for dc in DC_VALUES:
    out = MODELS_DIR / f'unsup_pooler_dc{dc}.pt'
    if out.exists():
        print(f'dc={dc}: existiert bereits.')
        continue
    cmd = [
        sys.executable,
        str(REPO_DIR / 'scripts' / 'train_unsupervised_pool.py'),
        '--embeddings', str(EMBED_OUT / 'deeploc_train.h5'),
        '--d', '1024', '--dc', str(dc),
        '--epochs', '5', '--batch-size', '32', '--lr', '1e-3',
        '--device', DEVICE, '--output', str(out),
    ]
    print(f'Trainiere Pooler dc={dc}...')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, cwd=str(REPO_DIR))
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Training fehlgeschlagen fuer dc={dc}')

## 13 · (Optional) PCA Covariance Pooler fitten

In [ ]:
for dc in DC_VALUES:
    out = MODELS_DIR / f'pca_pooler_dc{dc}.pt'
    if out.exists():
        print(f'dc={dc}: existiert bereits.')
        continue
    cmd = [
        sys.executable,
        str(REPO_DIR / 'scripts' / 'fit_pca_pool.py'),
        '--embeddings', str(EMBED_OUT / 'deeploc_train.h5'),
        '--d', '1024', '--dc', str(dc),
        '--output', str(out),
    ]
    print(f'Fitte PCA Pooler dc={dc}...')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, cwd=str(REPO_DIR))
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'PCA-Fit fehlgeschlagen fuer dc={dc}')

## 14 · Nächste Schritte (lokal)

HDF5-Dateien aus Drive in `data/embeddings/` kopieren, dann lokal:

```
conda activate sop

python scripts/run_experiment.py --config configs/scl/mean.yaml
python scripts/run_experiment.py --config configs/scl/cov_supervised.yaml --dc 8 16 24 32 48
python scripts/run_experiment.py --config configs/meltome/mean.yaml
```